# LeetCode 3261 — Count Substrings That Satisfy K-Constraint II bằng Python

- Link: https://leetcode.com/problems/count-substrings-that-satisfy-k-constraint-ii/
- Độ khó: **Hard**
- Chủ đề: **Hai con trỏ** tính `right[i]` (đầu phải xa nhất hợp lệ, **không giảm**), **tổng tiền tố** độ dài, **tìm kiếm nhị phân** điểm mà `right[i]` vượt `r` ⇒ mỗi truy vấn `O(log n)`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countKConstraintSubstrings(self, s: str, k: int, queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Xâu thỏa **k-constraint** nếu số `0` ≤ `k` **hoặc** số `1` ≤ `k`. Tính chất này **đóng với xâu con** (xâu con của xâu hợp lệ vẫn hợp lệ).
2. Do đó với mỗi đầu trái `i`, các đầu phải hợp lệ là đoạn `[i, right[i]]`; `right[i]` **không giảm** theo `i` ⇒ tính bằng hai con trỏ `O(n)`.
3. Truy vấn `[l, r]`: với `i ∈ [l, r]`, số xâu con bắt đầu tại `i` là `min(right[i], r) − i + 1`. Tách tại `p` = chỉ số đầu tiên có `right[p] ≥ r`: phần trước dùng tổng tiền tố của `right[i] − i + 1`, phần sau là cấp số cộng.
4. `p` tìm bằng `bisect` vì `right` không giảm.
5. Kiểm chứng bằng vét cạn.

## 2. Đề bài & ví dụ

Cho xâu nhị phân `s`, số `k` và các truy vấn `[l, r]`. Trả về, với mỗi truy vấn, số xâu con của `s[l..r]` thỏa k-constraint.

**Ví dụ 1:** `s = "0001111"`, `k = 2`, `[[0,6]]` → `[26]`.
**Ví dụ 2:** `s = "010101"`, `k = 1`, `[[0,5],[1,4],[2,3]]` → `[15,9,3]`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`; `1 ≤ k ≤ n`; `1 ≤ q ≤ 10^5`.

In [1]:
from typing import List
import bisect

examples = [("0001111", 2, [[0, 6]], [26]), ("010101", 1, [[0, 5], [1, 4], [2, 3]], [15, 9, 3])]
for s, k, qs, exp in examples:
    print(repr(s), 'k =', k, qs, '-> kỳ vọng', exp)

'0001111' k = 2 [[0, 6]] -> kỳ vọng [26]
'010101' k = 1 [[0, 5], [1, 4], [2, 3]] -> kỳ vọng [15, 9, 3]


## 3. Kiến thức nền

### 3.1. `right[i]` bằng hai con trỏ

Duyệt đầu phải `j`, giữ đầu trái nhỏ nhất `l` sao cho `s[l..j]` hợp lệ; khi đầu trái `i` bị đẩy qua (vì `s[i..j]` không hợp lệ), ta biết `right[i] = j − 1`. Cuối cùng các `i` còn lại có `right[i] = n − 1`.

In [2]:
def compute_right(s, k):
    n = len(s)
    right = [n - 1] * n
    c = [0, 0]
    l = 0
    for j, ch in enumerate(s):
        c[ord(ch) - 48] += 1
        while c[0] > k and c[1] > k:
            right[l] = j - 1
            c[ord(s[l]) - 48] -= 1
            l += 1
    return right

print('right("0001111", 2) =', compute_right("0001111", 2))

right("0001111", 2) = [4, 6, 6, 6, 6, 6, 6]


### 3.2. Tổng tiền tố

`P[i] = Σ_{t < i} (right[t] − t + 1)` — tổng số xâu con hợp lệ bắt đầu trước `i` (không giới hạn đầu phải).

### 3.3. Tổng cấp số cộng

`Σ_{i=p}^{r} (r − i + 1) = 1 + 2 + … + (r − p + 1) = L(L + 1)/2` với `L = r − p + 1`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `right = [4, 6, 6, 6, 6, 6, 6]` (từ `i = 0`, `"00011"` hợp lệ — hai số `1` — nhưng `"000111"` có ba `0` và ba `1`). Truy vấn `[0, 6]`:

- `p` = chỉ số đầu tiên có `right ≥ 6` = `1`.
- `i = 0`: `right[0] − 0 + 1 = 5`.
- `i ∈ [1, 6]`: `L = 6` ⇒ `21`.
- Tổng `26` ✓.

## 5. Cách 1 — Brute Force: đếm mọi xâu con trong mỗi truy vấn

In [3]:
def brute(s, k, queries):
    out = []
    for l, r in queries:
        cnt = 0
        for i in range(l, r + 1):
            z = o = 0
            for j in range(i, r + 1):
                if s[j] == '0':
                    z += 1
                else:
                    o += 1
                if z <= k or o <= k:
                    cnt += 1
        out.append(cnt)
    return out

for s, k, qs, exp in examples:
    print(brute(s, k, qs), '| kỳ vọng', exp)

[26] | kỳ vọng [26]
[15, 9, 3] | kỳ vọng [15, 9, 3]


**Độ phức tạp:** `O(q · n²)`.

## 6. Ý tưởng thuật toán / bước đột phá

```
right = compute_right(s, k);  P = tiền tố của (right[i] − i + 1)
answer(l, r):
    p = max(l, bisect_left(right, r))      # right không giảm
    p = min(p, r + 1)
    return (P[p] − P[l]) + L(L+1)/2  với L = r − p + 1
```

Tiền xử lý `O(n)`, mỗi truy vấn `O(log n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Hai con trỏ cho từng truy vấn (`O(q · n)`)

In [4]:
def per_query_two_pointers(s, k, queries):
    out = []
    for L, R in queries:
        c = [0, 0]; l = L; cnt = 0
        for j in range(L, R + 1):
            c[ord(s[j]) - 48] += 1
            while c[0] > k and c[1] > k:
                c[ord(s[l]) - 48] -= 1; l += 1
            cnt += j - l + 1
        out.append(cnt)
    return out

for s, k, qs, exp in examples:
    print(per_query_two_pointers(s, k, qs), '| kỳ vọng', exp)

[26] | kỳ vọng [26]
[15, 9, 3] | kỳ vọng [15, 9, 3]


Mỗi `j` cộng số đầu trái hợp lệ `j − l + 1` (tương tự LC 3258 — bản I).

### 7.2. Cách 3 — `right[]` + tổng tiền tố + bisect (lời giải chính)

In [5]:
def right_prefix_bisect(s, k, queries):
    n = len(s)
    right = compute_right(s, k)
    P = [0] * (n + 1)
    for i in range(n):
        P[i + 1] = P[i] + right[i] - i + 1
    out = []
    for l, r in queries:
        p = bisect.bisect_left(right, r, l, r + 1)
        L = r - p + 1
        out.append(P[p] - P[l] + L * (L + 1) // 2)
    return out

for s, k, qs, exp in examples:
    print(right_prefix_bisect(s, k, qs), '| kỳ vọng', exp)

[26] | kỳ vọng [26]
[15, 9, 3] | kỳ vọng [15, 9, 3]


`bisect_left(right, r, l, r + 1)` tìm trong đoạn `[l, r + 1)` chỉ số đầu tiên có `right ≥ r`; nếu không có, trả `r + 1` (`L = 0`). Luôn có `right[r] ≥ r` nên thực ra `p ≤ r`.

### 7.3. Cách 4 — Đối xứng: `left[j]` + bisect theo đầu phải

`left[j]` = đầu trái nhỏ nhất để `s[left[j]..j]` hợp lệ (không giảm theo `j`). Với `j ∈ [l, r]`, số xâu con kết thúc tại `j` trong truy vấn là `j − max(left[j], l) + 1`. Tách tại `p'` = chỉ số cuối cùng có `left[j] < l`.

In [6]:
def left_prefix_bisect(s, k, queries):
    n = len(s)
    left = [0] * n
    c = [0, 0]; l = 0
    for j, ch in enumerate(s):
        c[ord(ch) - 48] += 1
        while c[0] > k and c[1] > k:
            c[ord(s[l]) - 48] -= 1; l += 1
        left[j] = l
    Q = [0] * (n + 1)
    for j in range(n):
        Q[j + 1] = Q[j] + j - left[j] + 1
    out = []
    for lq, rq in queries:
        p = bisect.bisect_left(left, lq, lq, rq + 1)      # j đầu tiên có left[j] >= lq
        Lc = p - lq                                         # j ∈ [lq, p): bị chặn bởi lq -> j - lq + 1
        out.append(Lc * (Lc + 1) // 2 + Q[rq + 1] - Q[p])
    return out

for s, k, qs, exp in examples:
    print(left_prefix_bisect(s, k, qs), '| kỳ vọng', exp)

[26] | kỳ vọng [26]
[15, 9, 3] | kỳ vọng [15, 9, 3]


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2.

In [7]:
def trace(s, k, queries):
    n = len(s); right = compute_right(s, k)
    P = [0] * (n + 1)
    for i in range(n):
        P[i + 1] = P[i] + right[i] - i + 1
    print('right =', right, '| P =', P)
    for l, r in queries:
        p = bisect.bisect_left(right, r, l, r + 1); L = r - p + 1
        print(f"[{l}, {r}]: p = {p}, phần đầu P[{p}] − P[{l}] = {P[p] - P[l]}, phần sau L = {L} -> {L * (L + 1) // 2} => {P[p] - P[l] + L * (L + 1) // 2}")
    print('vét cạn:', brute(s, k, queries))

trace(*examples[1][:3])

right = [2, 3, 4, 5, 5, 5] | P = [0, 3, 6, 9, 12, 14, 15]
[0, 5]: p = 3, phần đầu P[3] − P[0] = 9, phần sau L = 3 -> 6 => 15
[1, 4]: p = 2, phần đầu P[2] − P[1] = 3, phần sau L = 3 -> 6 => 9
[2, 3]: p = 2, phần đầu P[2] − P[2] = 0, phần sau L = 2 -> 3 => 3
vét cạn: [15, 9, 3]


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def countKConstraintSubstrings(self, s: str, k: int, queries: List[List[int]]) -> List[int]:
        n = len(s)
        right = [n - 1] * n
        c0 = c1 = 0
        l = 0
        for j in range(n):
            if s[j] == '0':
                c0 += 1
            else:
                c1 += 1
            while c0 > k and c1 > k:
                right[l] = j - 1
                if s[l] == '0':
                    c0 -= 1
                else:
                    c1 -= 1
                l += 1
        P = [0] * (n + 1)
        for i in range(n):
            P[i + 1] = P[i] + right[i] - i + 1
        ans = []
        for lq, rq in queries:
            p = bisect.bisect_left(right, rq, lq, rq + 1)
            L = rq - p + 1
            ans.append(P[p] - P[lq] + L * (L + 1) // 2)
        return ans

for s, k, qs, exp in examples:
    got = Solution().countKConstraintSubstrings(s, k, qs)
    print(got, 'OK' if got == exp else 'SAI')

[26] OK
[15, 9, 3] OK


## 10. Vì sao thuật toán đúng

1. **Đóng với xâu con:** bỏ ký tự chỉ làm giảm số `0`/`1` ⇒ hợp lệ giữ nguyên. Vậy với `i` cố định, tập đầu phải hợp lệ là `[i, right[i]]`, và `right[i] ≤ right[i + 1]` (nếu `s[i..j]` hợp lệ thì `s[i+1..j]` cũng vậy).
2. **Hai con trỏ:** khi `s[l..j]` không hợp lệ (cả hai số đếm `> k`), mọi `s[l..j']` với `j' ≥ j` không hợp lệ, còn `s[l..j−1]` hợp lệ (bất biến trước khi thêm `s[j]`) ⇒ `right[l] = j − 1`.
3. **Truy vấn:** số xâu con của `s[l..r]` bắt đầu tại `i` là `min(right[i], r) − i + 1`. Do `right` không giảm, `right[i] < r` đúng với `i < p` và `≥ r` với `i ≥ p` — hai phần tính bằng tổng tiền tố và công thức cấp số cộng.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(q n²)` | `O(1)` |
| 2. Hai con trỏ mỗi truy vấn | `O(q n)` | `O(1)` |
| 3. `right` + tiền tố + bisect | `O(n + q log n)` | `O(n)` |
| 4. `left` + tiền tố + bisect | `O(n + q log n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'per_query': per_query_two_pointers, 'right_bisect': right_prefix_bisect, 'left_bisect': left_prefix_bisect,
         'submit': lambda s, k, q: Solution().countKConstraintSubstrings(s, k, q)}
edge = [("0", 1, [[0, 0]], [1]), ("01", 1, [[0, 1]], [3]), ("0011", 1, [[0, 3], [1, 2]], [9, 3]),
        ("1111", 1, [[0, 3]], [10]), ("000111", 3, [[0, 5]], [21])]
for s, k, qs, exp in examples + edge:
    assert brute(s, k, qs) == exp, (s, k, qs, brute(s, k, qs))
    for name, f in impls.items():
        assert f(s, k, qs) == exp, (name, s, k, qs)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [10]:
import random
rng = random.Random(3261)
cnt = 0
for _ in range(500):
    n = rng.randint(1, 14)
    s = ''.join(rng.choice('01') for _ in range(n))
    k = rng.randint(1, n)
    qs = []
    for _ in range(rng.randint(1, 6)):
        l = rng.randrange(n); r = rng.randrange(l, n); qs.append([l, r])
    exp = brute(s, k, qs)
    for name, f in impls.items():
        assert f(s, k, qs) == exp, (name, s, k, qs)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test.')

Khớp vét cạn trên 500 bộ test.


### 12.3. Ca lớn `n = q = 10^5`

In [11]:
import time
N = 10 ** 5
for name, s, k in [('ngẫu nhiên, k = 3', ''.join(rng.choice('01') for _ in range(N)), 3),
                   ('toàn 0, k = 1', '0' * N, 1),
                   ('xen kẽ, k = 1000', '01' * (N // 2), 1000)]:
    qs = []
    for _ in range(N):
        l = rng.randrange(N); r = rng.randrange(l, N); qs.append([l, r])
    t = time.time(); a = Solution().countKConstraintSubstrings(s, k, qs); t1 = time.time() - t
    t = time.time(); b = left_prefix_bisect(s, k, qs); t2 = time.time() - t
    assert a == b
    print(f"{name:20s}: nộp {t1:.2f}s | cách 4 {t2:.2f}s")
qs = [[0, N - 1], [5, 100]]
assert Solution().countKConstraintSubstrings('0' * N, 1, qs) == [N * (N + 1) // 2, 96 * 97 // 2]
print('toàn 0: mọi xâu con hợp lệ (số 1 = 0 <= k) — đúng.')

ngẫu nhiên, k = 3   : nộp 0.11s | cách 4 0.13s


toàn 0, k = 1       : nộp 0.08s | cách 4 0.08s


xen kẽ, k = 1000    : nộp 0.13s | cách 4 0.12s
toàn 0: mọi xâu con hợp lệ (số 1 = 0 <= k) — đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng "và" thay vì "hoặc"

In [12]:
def wrong_and(s, k, queries):
    out = []
    for l, r in queries:
        out.append(sum(1 for i in range(l, r + 1) for j in range(i, r + 1)
                       if s[i:j + 1].count('0') <= k and s[i:j + 1].count('1') <= k))     # SAI
    return out

print('sai :', wrong_and(*examples[0][:3]), '| đúng:', Solution().countKConstraintSubstrings(*examples[0][:3]))

sai : [16] | đúng: [26]


**Vì sao sai:** chỉ cần **một** trong hai điều kiện (`"0001"` có 3 số `0` > 2 nhưng chỉ 1 số `1` ≤ 2 ⇒ hợp lệ).

### Lỗi 2 — Không chặn `right[i]` bởi `r`

In [13]:
def wrong_no_cap(s, k, queries):
    right = compute_right(s, k)
    return [sum(right[i] - i + 1 for i in range(l, r + 1)) for l, r in queries]   # SAI

print('sai :', wrong_no_cap(*examples[1][:3]), '| đúng:', Solution().countKConstraintSubstrings(*examples[1][:3]))

sai : [15, 11, 6] | đúng: [15, 9, 3]


**Vì sao sai:** xâu con phải nằm **trong** `s[l..r]`; `right[i]` có thể vượt `r`.

### Lỗi 3 — Tìm `p` trên **toàn mảng** (quên giới hạn `[l, r + 1)`)

In [14]:
def wrong_bisect_right(s, k, queries):
    n = len(s); right = compute_right(s, k)
    P = [0] * (n + 1)
    for i in range(n):
        P[i + 1] = P[i] + right[i] - i + 1
    out = []
    for l, r in queries:
        p = bisect.bisect_left(right, r)                            # SAI: p có thể < l
        L = r - p + 1
        out.append(P[p] - P[l] + L * (L + 1) // 2)
    return out

print('sai :', wrong_bisect_right("0000", 1, [[1, 2]]), '| đúng:', Solution().countKConstraintSubstrings("0000", 1, [[1, 2]]))

sai : [2] | đúng: [3]


**Vì sao sai:** nếu đã có `right[i] ≥ r` với `i < l`, `p` rơi về bên trái `l`: phần "trước" thành âm (`P[p] − P[l] < 0`) còn phần cấp số cộng đếm cả các đầu trái `i < l`. Điểm tách phải nằm trong `[l, r + 1]`. (Lưu ý: dùng `bisect_right` thay `bisect_left` thì **không** sai — các `i` có `right[i] = r` cho cùng giá trị ở cả hai phần.)

### Lỗi 4 — Trả lời bằng hiệu "đáp án tiền tố" `F(r) − F(l − 1)`

In [15]:
def wrong_prefix_difference(s, k, queries):
    n = len(s)
    F = per_query_two_pointers(s, k, [[0, r] for r in range(n)])   # F[r] = đáp án cho [0, r]
    return [F[r] - (F[l - 1] if l else 0) for l, r in queries]       # SAI

print('sai :', wrong_prefix_difference(*examples[1][:3]), '| đúng:', Solution().countKConstraintSubstrings(*examples[1][:3]))

sai : [15, 11, 6] | đúng: [15, 9, 3]


**Vì sao sai:** `F(r) − F(l − 1)` còn chứa các xâu con **bắt đầu trước `l`** và kết thúc trong `[l, r]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Tính chất "đóng với xâu con" ⇒ mỗi đầu trái có một đoạn đầu phải hợp lệ; ranh giới đơn điệu ⇒ hai con trỏ.
- Truy vấn đoạn trên tổng `min(f(i), r) − i + 1` với `f` đơn điệu ⇒ bisect + tổng tiền tố + công thức đóng.
- Không dùng "hiệu tiền tố" khi đối tượng đếm là **đoạn con** (bị cắt hai đầu).

## 15. Biến thể của bài

1. **LC 3258 (bản I):** một truy vấn trên cả xâu.
2. Điều kiện "**cả hai** số đếm ≤ k" — vẫn đóng với xâu con, cùng kỹ thuật.
3. Bảng chữ cái lớn hơn với điều kiện "có một ký tự (trong bảng chữ cho trước) xuất hiện ≤ k lần" — vẫn đóng với xâu con nên vẫn dùng hai con trỏ.
4. Đếm xâu con **không** thỏa k-constraint ⇒ tổng `L(L+1)/2` trừ đáp án.

In [16]:
# Biến thể 2: cả hai số đếm <= k
def both_variant(s, k, queries):
    n = len(s); right = [n - 1] * n; c = [0, 0]; l = 0
    for j, ch in enumerate(s):
        c[ord(ch) - 48] += 1
        while c[0] > k or c[1] > k:
            right[l] = j - 1; c[ord(s[l]) - 48] -= 1; l += 1
    P = [0] * (n + 1)
    for i in range(n):
        P[i + 1] = P[i] + right[i] - i + 1
    out = []
    for lq, rq in queries:
        p = bisect.bisect_left(right, rq, lq, rq + 1); L = rq - p + 1
        out.append(P[p] - P[lq] + L * (L + 1) // 2)
    return out

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 12); s = ''.join(rng.choice('01') for _ in range(n)); k = rng.randint(1, n)
    l = rng.randrange(n); r = rng.randrange(l, n)
    assert both_variant(s, k, [[l, r]]) == wrong_and(s, k, [[l, r]])
print('Biến thể "cả hai <= k" (chính là phiên bản "và") khớp vét cạn.')

Biến thể "cả hai <= k" (chính là phiên bản "và") khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Vét cạn | `O(q n²)` | Oracle |
| 2 | Hai con trỏ mỗi truy vấn | `O(q n)` | Bản I |
| 3 | `right` + bisect | `O(n + q log n)` | **Nộp bài** |
| 4 | `left` + bisect | `O(n + q log n)` | Góc nhìn đối xứng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "1"`, `k = 1`, `[[0, 0]]`.

**Bài 2 (tính tay):** `s = "0101"`, `k = 1`, `[[0, 3]]`.

**Bài 3 (điền TODO):** đếm xâu con **không** thỏa trong mỗi truy vấn.

**Bài 4 (tự viết lại):** độ dài xâu con hợp lệ **dài nhất** trong mỗi truy vấn.

**Bài 5 (mini challenge):** truy vấn trực tuyến khi `k` thay đổi theo truy vấn (`k` nhỏ, ≤ 5) — tiền xử lý `right` cho mỗi `k`.

In [17]:
def invalid_count_todo(s, k, queries):
    # TODO
    raise NotImplementedError

def longest_valid_todo(s, k, queries):
    # TODO
    raise NotImplementedError

def multi_k_todo(s, queries_with_k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một xâu con. (2) Hợp lệ. (3) `[1]`.

**Bài 2.** (1) Xâu con độ dài ≤ 3 luôn hợp lệ với `k = 1`? `"010"`: hai số 0, một số 1 ⇒ hợp lệ. (2) Độ dài 4 `"0101"`: hai và hai ⇒ không. (3) `10 − 1 = 9`.

**Bài 3.** (1) `L(L+1)/2` với `L = r − l + 1`. (2) Trừ đáp án. (3) Kiểm vét cạn.

**Bài 4.** (1) `max_{i ∈ [l, r]} (min(right[i], r) − i + 1)`. (2) Phần `i ≥ p`: tốt nhất là `i = p` ⇒ `r − p + 1`. (3) Phần `i < p`: max của `right[i] − i + 1` — sparse table.

**Bài 5.** (1) `right_k` cho mỗi `k ≤ 5`. (2) Chọn bảng theo `k` của truy vấn. (3) `O(5n + q log n)`.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().countKConstraintSubstrings("1", 1, [[0, 0]]) == [1]
assert Solution().countKConstraintSubstrings("0101", 1, [[0, 3]]) == [9]

def invalid_count(s, k, queries):
    valid = Solution().countKConstraintSubstrings(s, k, queries)
    return [(r - l + 1) * (r - l + 2) // 2 - v for (l, r), v in zip(queries, valid)]

assert invalid_count("0001111", 2, [[0, 6]]) == [2]
print('Bài 1: [1] | Bài 2: [9] | Bài 3:', invalid_count("0001111", 2, [[0, 6]]))

Bài 1: [1] | Bài 2: [9] | Bài 3: [2]


In [19]:
# Bài 4, 5
def longest_valid(s, k, queries):
    n = len(s); right = compute_right(s, k)
    vals = [right[i] - i + 1 for i in range(n)]
    LOG = max(1, n.bit_length()); sp = [vals[:]]
    for j in range(1, LOG):
        prev = sp[-1]; step = 1 << (j - 1)
        sp.append([max(prev[i], prev[i + step]) for i in range(n - (1 << j) + 1)])
    def rmq(a, b):                      # max vals[a..b-1], a < b
        j = (b - a).bit_length() - 1
        return max(sp[j][a], sp[j][b - (1 << j)])
    out = []
    for l, r in queries:
        p = bisect.bisect_left(right, r, l, r + 1)
        best = r - p + 1 if p <= r else 0
        if p > l:
            best = max(best, rmq(l, p))
        out.append(best)
    return out

def multi_k(s, queries_with_k):
    out = []
    for l, r, k in queries_with_k:
        out.append(Solution().countKConstraintSubstrings(s, k, [[l, r]])[0])
    return out

rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 12); s = ''.join(rng.choice('01') for _ in range(n)); k = rng.randint(1, n)
    l = rng.randrange(n); r = rng.randrange(l, n)
    exp = max(j - i + 1 for i in range(l, r + 1) for j in range(i, r + 1)
              if s[i:j + 1].count('0') <= k or s[i:j + 1].count('1') <= k)
    assert longest_valid(s, k, [[l, r]]) == [exp]
    qk = [(l, r, rng.randint(1, 3))]
    assert multi_k(s, qk) == [brute(s, qk[0][2], [[l, r]])[0]]
print('Bài 4 (sparse table) và Bài 5 khớp vét cạn.')

Bài 4 (sparse table) và Bài 5 khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
right[i] bằng hai con trỏ (gán khi đầu trái bị đẩy qua); mặc định n-1
P[i+1] = P[i] + right[i] - i + 1
p = bisect_left(right, r, l, r+1); L = r - p + 1
ans = P[p] - P[l] + L*(L+1)//2
```

**Checklist:**

- [ ] Điều kiện là "số 0 ≤ k **hoặc** số 1 ≤ k"?
- [ ] Không hợp lệ khi **cả hai** số đếm `> k` (điều kiện vòng `while`)?
- [ ] `right[i]` mặc định `n − 1`?
- [ ] Chặn bởi `r` qua điểm tách `p`?
- [ ] Không dùng hiệu đáp án tiền tố?